In [ ]:
import json
json_path = "Datasets/test_2_w_spec.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)
# =========================
# Collect unique topo pairs
# =========================
topo_counter = Counter()

for item in data:
    spec = item.get("extended_spec", {})
    topo = spec.get("topo", {})

    t = topo.get("type", "")
    sub = topo.get("sub", "")

    # Normalize type → list
    if isinstance(t, list):
        types = [str(x).lower() for x in t]
    elif isinstance(t, str):
        types = [t.lower()]
    else:
        types = ["unknown"]

    sub = sub.lower() if isinstance(sub, str) else ""

    for tt in types:
        topo_counter[(tt, sub)] += 1

# =========================
# Print results
# =========================
print("Unique (type, sub) pairs:")
print("-" * 50)

for (t, s), cnt in sorted(topo_counter.items(), key=lambda x: -x[1]):
    print(f"type: {t:<15} | sub: {s:<15} | count: {cnt}")

print("-" * 50)
print(f"Total unique topo pairs: {len(unique_topo)}")

In [ ]:
import json
import pandas as pd
from collections import Counter

# =========================
# Load Data
# =========================
csv_path = "results/chartcheck_qualitative_results.csv"
json_path = "Datasets/test_1_w_spec.json"

df = pd.read_csv(csv_path)

# --- FILTER: Only Test 1 ---
df = df[df["Split"] == "Test 1"].copy()
print(len(df))


# Load JSON spec
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

spec_map = {
    item["claim"]: item["extended_spec"]
    for item in data
}

# =========================
# Your TOPO MAP (with restriction)
# =========================
_TOPO_MAP = {
    ("bar",  "vertical"):   "barchart_vertical",
    ("bar",  "horizontal"): "barchart_horizontal",
    ("bar",  "stacked"):    "barchart_vertical",
    ("bar",  ""):           "barchart_vertical",

    ("line", ""):           "line_chart",
    ("line", "area"):       "line_chart",

    ("pie",  ""):           "pie_chart",
    ("pie",  "donut"):      "pie_chart",

    ("scatter", ""):        "scatter_plot",

    # These will be collapsed later (not in your allowed set)
    ("bubble",  ""):        "bubble_chart",
    ("map",     ""):        "map_chart",
}

# Allowed final label space
_ALLOWED = {
    "pie_chart",
    "line_chart",
    "barchart_vertical",
    "barchart_horizontal",
    "histogram",
    "unknown",
    "bar+line",
    "area",
    "scatter_plot",
}

# =========================
# Mapping Function (robust)
# =========================
def map_spec_to_fixed_label(spec, csv_label):
    topo = spec.get("topo", {})
    
    t = topo.get("type", "")
    sub = topo.get("sub", "")

    # Normalize
    if isinstance(t, list):
        types = [str(x).lower() for x in t]
    elif isinstance(t, str):
        types = [t.lower()]
    else:
        types = []

    sub = sub.lower() if isinstance(sub, str) else ""

    # =========================
    # HARD FALLBACK (your rule)
    # =========================
    if (not types or types == [""]) and sub == "":
        return csv_label

    # =========================
    # COMBO CASES
    # =========================
    if "bar+line" in types or ("bar" in types and "line" in types):
        return "bar+line"

    if "area+line" in types:
        return "area"

    # =========================
    # PIE FAMILY
    # =========================
    if "pie" in types:
        return "pie_chart"

    if "circle" in types and sub == "donut":
        return "pie_chart"

    # =========================
    # BAR
    # =========================
    if "bar" in types:
        if sub == "horizontal":
            return "barchart_horizontal"
        return "barchart_vertical"

    # =========================
    # LINE
    # =========================
    if "line" in types:
        return "line_chart"

    # =========================
    # HISTOGRAM
    # =========================
    if "histogram" in types:
        return "histogram"

    # =========================
    # AREA
    # =========================
    if "area" in types:
        return "area"

    # =========================
    # SCATTER
    # =========================
    if "scatter" in types:
        return "scatter_plot"

    # =========================
    # FINAL FALLBACK
    # =========================
    return csv_label


def final_chart_type(row):
    spec = spec_map.get(row["Claim"], None)

    if spec is None:
        return row["Chart Type"]
    

    return map_spec_to_fixed_label(spec, row["Chart Type"])


# =========================
# Apply Mapping
# =========================
df["Mapped Chart Type"] = df.apply(final_chart_type, axis=1)

# =========================
# Fine-grain print function (RETAINED)
# =========================
def print_fine_grain_accuracy(df, chart_col="Mapped Chart Type"):
    print("\nPer-Chart-Type Accuracy")
    print("-" * 60)

    results = []

    for chart_type, group in df.groupby(chart_col):
        total = len(group)
        correct = (group["Predicted Verdict"] == group["True Verdict"]).sum()
        acc = (correct / total) * 100 if total > 0 else 0

        results.append((chart_type, acc, total))

    # sort by count (like your previous output)
    results = sorted(results, key=lambda x: -x[2])

    for chart_type, acc, total in results:
        print(f"{chart_type:<30} {acc:>6.1f}%    {total}")

    print("-" * 60)
    print(f"Total samples accounted for: {len(df)} / {len(df)}")


print("\n--- Original Labels ---")
print_fine_grain_accuracy(df, chart_col="Chart Type")

print("\n--- Mapped Labels ---")
print_fine_grain_accuracy(df, chart_col="Mapped Chart Type")

# =========================
# Debug: changed rows
# =========================
changed = df[df["Chart Type"] != df["Mapped Chart Type"]]

print(f"\nChanged rows: {len(changed)} / {len(df)}")
display(changed.head(10))